# Implement Persistent Memory Agent

This notebook creates an Azure AI Foundry agent that can remember user context across sessions and notebook restarts.

Unlike the session-aware assistant, this demo stores long-term user facts in an Azure AI Foundry Memory Store. Each user gets a separate memory `scope`, so one learner's preferences and facts do not leak into another learner's chat.

## 1. Install required packages

Run this once if your environment does not already have these packages.

In [ ]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

## 2. Load Azure AI Foundry configuration

The notebook looks for `.env` in this order:

1. This notebook folder
2. `../A2A_and_MCP/.env`
3. `../Getting_Started_Foundry_Agent/.env`
4. `../../AgentService/Memories/.env`

Expected values:

```text
FOUNDRY_PROJECT_ENDPOINT="https://...services.ai.azure.com/api/projects/..."
MODEL_DEPLOYMENT_NAME="your-chat-model-deployment-name"
TEXT_EMBEDDING_MODEL_NAME="your-embedding-model-deployment-name"
```

`TEXT_EMBEDDING_MODEL_NAME` should match an embedding deployment in your Foundry project. If it is not set, this notebook uses `text-embedding-3-small` as a demo default; change that default if your deployment has another name.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()

candidate_env_paths = [
    cwd / ".env",
    cwd / "A2A" / "Memory-Build session-aware assistant" / ".env",
    cwd / "Memory-Build session-aware assistant" / ".env",
    cwd / "A2A_and_MCP" / ".env",
    cwd / "Getting_Started_Foundry_Agent" / ".env",
    cwd / "AgentService" / "Memories" / ".env",
    cwd.parent / "A2A_and_MCP" / ".env",
    cwd.parent / "Getting_Started_Foundry_Agent" / ".env",
    cwd.parent.parent / "AgentService" / "Memories" / ".env",
]

loaded_env_paths = []
for env_path in candidate_env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        loaded_env_paths.append(env_path)

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

# Change this if your embedding deployment uses a custom name in Foundry.
default_embedding_deployment_name = "text-embedding-3-small"
embedding_model_name = (
    os.getenv("TEXT_EMBEDDING_MODEL_NAME")
    or os.getenv("EMBEDDING_MODEL_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
    or default_embedding_deployment_name
)

missing = [
    name
    for name, value in {
        "FOUNDRY_PROJECT_ENDPOINT": foundry_project_endpoint,
        "MODEL_DEPLOYMENT_NAME": model_deployment_name,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        "Missing required environment variable(s): "
        + ", ".join(missing)
        + ". Add them to a .env file and rerun this cell."
    )

print("Loaded configuration from:")
for env_path in loaded_env_paths:
    print(f"- {env_path}")
print(f"Foundry project endpoint: {foundry_project_endpoint}")
print(f"Chat model deployment: {model_deployment_name}")
print(f"Embedding model deployment: {embedding_model_name}")
if embedding_model_name == default_embedding_deployment_name:
    print(
        "Using demo default embedding deployment. "
        "If your Foundry deployment has a different name, set TEXT_EMBEDDING_MODEL_NAME in .env."
    )

## 3. Connect to the Azure AI Foundry project

This uses your Azure sign-in through `DefaultAzureCredential`. If authentication fails, run `az login` in a terminal and rerun this cell.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()

client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=credential,
)

openai_client = client.get_openai_client()
print("Connected to Azure AI Foundry project.")

## 4. Create or reuse a Memory Store

The Memory Store is the persistent layer. The store remains available in the Foundry project after this notebook kernel stops.

This demo enables:

- `user_profile_enabled`: extracts durable facts about a user.
- `chat_summary_enabled`: summarizes relevant conversational context.

In [ ]:
from azure.core.exceptions import ResourceNotFoundError, ResourceExistsError
from azure.ai.projects.models import MemoryStoreDefaultDefinition, MemoryStoreDefaultOptions

memory_store_name = "persistent_memory_agent_store_v1"

memory_definition = MemoryStoreDefaultDefinition(
    chat_model=model_deployment_name,
    embedding_model=embedding_model_name,
    options=MemoryStoreDefaultOptions(
        user_profile_enabled=True,
        chat_summary_enabled=True,
    ),
)


def get_or_create_memory_store(name: str):
    try:
        store = client.memory_stores.get(name=name)
        print(f"Using existing Memory Store: {store.name}")
        return store
    except ResourceNotFoundError:
        pass

    try:
        store = client.memory_stores.create(
            name=name,
            definition=memory_definition,
            description="Persistent memory store for the Foundry persistent memory agent demo",
        )
        print(f"Created Memory Store: {store.name}")
        return store
    except ResourceExistsError:
        store = client.memory_stores.get(name=name)
        print(f"Memory Store already exists, using: {store.name}")
        return store


memory_store = get_or_create_memory_store(memory_store_name)

## 5. Create the persistent memory agent

The agent answers normally, but the wrapper class below retrieves user-specific memories and injects them as grounding context before each response.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent_name = "persistent-memory-agent"

agent_instructions = """
You are a persistent-memory training assistant for Azure AI Foundry learners.
Use the supplied long-term memory context when it is relevant.
Do not invent memories that were not supplied.
If no relevant memory is supplied, say you do not have that detail yet.
Keep answers concise, practical, and demo-friendly.
"""

agent = client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions=agent_instructions,
    ),
)

print(f"Agent created: id={agent.id}, name={agent.name}, version={agent.version}")

## 6. Build the persistent memory wrapper

`PersistentMemoryAgent` keeps long-term memories in the Foundry Memory Store using `user_id` as the scope.

That means:

- Same `user_id` -> same long-term memory scope.
- Different `user_id` -> isolated memories.
- Restarting the notebook does not erase the stored memories.

In [ ]:
from azure.ai.projects.models import MemorySearchOptions, ResponsesUserMessageItemParam


class PersistentMemoryAgent:
    def __init__(self, client: AIProjectClient, memory_store_name: str, agent_name: str):
        self.client = client
        self.memory_store_name = memory_store_name
        self.agent_name = agent_name
        self.openai_client = client.get_openai_client()

    def remember(self, user_id: str, text: str):
        item = ResponsesUserMessageItemParam(content=text)

        poller = self.client.memory_stores.begin_update_memories(
            name=self.memory_store_name,
            scope=user_id,
            items=[item],
            update_delay=0,
        )
        result = poller.result()

        operations = getattr(result, "memory_operations", []) or []
        print(f"Memory update completed for '{user_id}' with {len(operations)} operation(s).")
        for operation in operations:
            memory_item = getattr(operation, "memory_item", None)
            content = getattr(memory_item, "content", "") if memory_item else ""
            print(f"- {operation.kind}: {content}")

        return result

    def search(self, user_id: str, query: str, max_memories: int = 5) -> list[str]:
        item = ResponsesUserMessageItemParam(content=query)

        search_result = self.client.memory_stores.search_memories(
            name=self.memory_store_name,
            scope=user_id,
            items=[item],
            options=MemorySearchOptions(max_memories=max_memories),
        )

        memories = []
        for memory in getattr(search_result, "memories", []) or []:
            memory_item = getattr(memory, "memory_item", None)
            content = getattr(memory_item, "content", None) if memory_item else None
            if content:
                memories.append(content)

        return memories

    def ask(self, user_id: str, user_message: str, auto_remember: bool = True) -> str:
        if auto_remember:
            self.remember(user_id=user_id, text=user_message)

        memories = self.search(user_id=user_id, query=user_message)
        memory_context = "\n".join(f"- {memory}" for memory in memories) or "No relevant long-term memories found."

        grounded_input = f"""
User id: {user_id}

Relevant long-term memories:
{memory_context}

User message:
{user_message}
""".strip()

        response = self.openai_client.responses.create(
            extra_body={
                "agent": {
                    "name": self.agent_name,
                    "type": "agent_reference",
                }
            },
            input=grounded_input,
        )

        return response.output_text

    def delete_user_memory(self, user_id: str):
        return self.client.memory_stores.delete_scope(
            name=self.memory_store_name,
            scope=user_id,
        )


persistent_agent = PersistentMemoryAgent(
    client=client,
    memory_store_name=memory_store.name,
    agent_name=agent_name,
)

print("Persistent memory agent is ready.")

## 7. Store long-term user facts

These facts are persisted in the Memory Store under the `learner-ajay` scope.

In [ ]:
persistent_agent.remember(
    user_id="learner-ajay",
    text=(
        "My name is Ajay. I am preparing an Agentic AI Level 3 training demo. "
        "I prefer Python examples and concise explanations."
    ),
)

## 8. Ask a question that requires memory

The agent should retrieve the stored facts and use them in the answer.

In [ ]:
reply = persistent_agent.ask(
    user_id="learner-ajay",
    user_message="What do you remember about me, and how should you tailor this demo?",
    auto_remember=False,
)

print(reply)

## 9. Prove memory isolation with another user

This second user has a different `user_id`, so the agent should not see Ajay's details.

In [ ]:
reply = persistent_agent.ask(
    user_id="learner-priya",
    user_message="What do you remember about my name and training preference?",
    auto_remember=False,
)

print(reply)

## 10. Add memory during a chat turn

With `auto_remember=True`, the wrapper updates the Memory Store before answering.

In [ ]:
reply = persistent_agent.ask(
    user_id="learner-priya",
    user_message="My name is Priya. I am focused on enterprise governance and responsible AI.",
    auto_remember=True,
)

print(reply)

In [ ]:
reply = persistent_agent.ask(
    user_id="learner-priya",
    user_message="Now what do you remember about my focus area?",
    auto_remember=False,
)

print(reply)

## 11. Inspect retrieved memories directly

This is useful during a live demo because you can show the memory retrieval step before the agent response.

In [ ]:
persistent_agent.search(
    user_id="learner-ajay",
    query="What training preferences should the assistant remember?",
    max_memories=5,
)

## 12. Optional cleanup

Use this only when you want to delete demo memory for a user scope. Deleting a scope removes that user's persisted memories from the Memory Store.

In [ ]:
# Optional cleanup for demo users.
# persistent_agent.delete_user_memory("learner-ajay")
# persistent_agent.delete_user_memory("learner-priya")

## Wiring Summary

```mermaid
flowchart LR
    User[User message] --> Scope[user_id memory scope]
    Scope --> Update[Optional update memories]
    Scope --> Search[Search Memory Store]
    Search --> Context[Relevant long-term memories]
    Context --> Prompt[Grounded agent input]
    User --> Prompt
    Prompt --> Agent[persistent-memory-agent]
    Agent --> Answer[Assistant response]
```

The important idea: session memory remembers a conversation ID, while persistent memory stores retrievable facts in a Memory Store scoped to a user.